# ADAS Lane2OpenDRIVE Colab

Run the cells from top to bottom on a Google Colab **T4 GPU**. This notebook uses the official Ultra-Fast-Lane-Detection CULane pretrained 2D lane detector for arbitrary videos. It never creates dummy detections or fabricated metric values.

Metric distance, calibrated visual-motion speed, and OpenDRIVE require real camera calibration values in `configs/camera.yaml`. Without them, the run safely produces image-space lanes and reports `NON_METRIC`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/vamos-sujal/lane-opendrive.git'
REPO_DIR = Path('/content/lane-opendrive')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('Repository:', Path.cwd())

In [ ]:
import subprocess
import sys
import torch

print(sys.version)
subprocess.run(['nvidia-smi'], check=True)
assert torch.cuda.is_available(), 'CUDA is unavailable. In Colab choose Runtime > Change runtime type > T4 GPU.'
gpu_name = torch.cuda.get_device_name(0)
print('GPU:', gpu_name, 'CUDA:', torch.version.cuda, 'PyTorch:', torch.__version__)
assert 'T4' in gpu_name, f'Expected a T4 GPU, got {gpu_name}. Select a T4 runtime and rerun from the first cell.'

In [ ]:
import os
import subprocess
from pathlib import Path

os.chdir('/content/lane-opendrive')
def run(command):
    print('$', command)
    subprocess.run(command, shell=True, check=True)

run('python -m pip install --upgrade pip')
run('python -m pip install -r /content/lane-opendrive/requirements.txt addict==2.4.0 pathspec==0.12.1')
run('rm -rf /content/Ultra-Fast-Lane-Detection')
run('git clone --depth 1 https://github.com/cfzd/Ultra-Fast-Lane-Detection.git /content/Ultra-Fast-Lane-Detection')
run('python /content/lane-opendrive/scripts/verify_environment.py')

import torch
assert torch.cuda.is_available(), 'CUDA is required for the T4 lane detector.'
print('UFLD source ready:', '/content/Ultra-Fast-Lane-Detection')
print('Environment verification passed.')

In [ ]:
import os
import subprocess
import yaml
from pathlib import Path

os.chdir('/content/lane-opendrive')
MODEL_DIR = Path('/content/drive/MyDrive/lane_to_opendrive/models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_config = yaml.safe_load(Path('/content/lane-opendrive/configs/model.yaml').read_text())
checkpoint_name = Path(model_config['checkpoint']['path']).name
CHECKPOINT_PATH = MODEL_DIR / checkpoint_name

if not CHECKPOINT_PATH.exists():
    subprocess.run([
        'python', '/content/lane-opendrive/scripts/download_weights.py',
        '--config', '/content/lane-opendrive/configs/model.yaml',
        '--output-dir', str(MODEL_DIR),
    ], check=True)
else:
    print('Reusing checkpoint from Drive:', CHECKPOINT_PATH)

assert CHECKPOINT_PATH.exists(), f'Checkpoint was not found: {CHECKPOINT_PATH}'
subprocess.run(['python', '/content/lane-opendrive/scripts/smoke_test.py', '--device', 'cuda'], check=True)
print('Loaded detector contract: official UFLD CULane pretrained checkpoint')

In [ ]:
from pathlib import Path
from google.colab import files

INPUT_DIR = Path('/content/drive/MyDrive/lane_to_opendrive/input')
RUNS_DIR = Path('/content/drive/MyDrive/lane_to_opendrive/runs')
MODEL_DIR = Path('/content/drive/MyDrive/lane_to_opendrive/models')
INPUT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_PATH = str(INPUT_DIR)
video_files = sorted(path for path in INPUT_DIR.iterdir() if path.suffix.lower() in {'.mp4', '.mov', '.avi'})
if len(video_files) == 0:
    print('No video found. Upload one video with the optional code below, then rerun this cell.')
    print('Accepted formats: .mp4, .mov, .avi')
else:
    print('Video selected:', video_files[0])
    if len(video_files) > 1 and not (INPUT_DIR / 'input.mp4').exists():
        raise RuntimeError('Multiple videos found. Keep one video or rename the intended file to input.mp4.')

# Optional upload workflow:
# uploaded = files.upload()
# for name in uploaded:
#     (INPUT_DIR / name).write_bytes(uploaded[name])
# VIDEO_PATH = str(INPUT_DIR)

In [ ]:
import datetime
import os
import subprocess

os.chdir('/content/lane-opendrive')
run_id = datetime.datetime.now(datetime.timezone.utc).strftime('run_%Y%m%dT%H%M%SZ')
RUN_DIR = RUNS_DIR / run_id
subprocess.run([
    'python', '/content/lane-opendrive/scripts/run_video.py',
    '--input', VIDEO_PATH,
    '--output', str(RUN_DIR),
    '--config', '/content/lane-opendrive/configs/camera.yaml',
    '--detector', 'ufld_culane',
    '--checkpoint', str(CHECKPOINT_PATH),
], check=True)

In [ ]:
import json
from IPython.display import display, Image, Video

summary_path = RUN_DIR / 'run_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print('=== VIDEO REPORT ===')
    print(json.dumps(summary, indent=2))
    print('\n=== INTERPRETATION ===')
    print('Lane count and IDs are detector/tracker outputs.')
    print('Metric distance and speed are shown only when trusted calibration is configured.')
    print('OpenDRIVE is emitted only after metric geometry and XML validation pass.')

for name in ('input_metadata.json', 'metric_report.json', 'tracking_results.json', 'lane_graph.json', 'geometry.json', 'validation_report.json'):
    path = RUN_DIR / name
    if path.exists():
        print(f'\n{name}')
        print(json.dumps(json.loads(path.read_text()), indent=2))

visual_dir = RUN_DIR / 'visualizations'
for name in ('original_video.mp4', 'lane_overlay.mp4', 'topology_overlay.mp4', 'top_down.mp4'):
    path = visual_dir / name
    if path.exists() and path.stat().st_size > 0:
        print(f'\n{name}')
        display(Video(filename=str(path), embed=False))

for path in sorted(visual_dir.glob('*_frame_*.jpg')):
    display(Image(filename=str(path)))

xodr = RUN_DIR / 'output.xodr'
print('\nOpenDRIVE:', xodr if xodr.exists() else 'not produced: metric validation failed closed')

In [ ]:
from IPython.display import display, Image
for image in sorted((RUN_DIR / 'visualizations').glob('*.jpg')):
    print(image.name)
    display(Image(filename=str(image)))